# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a practical guide for loading and exploring a FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets using their @id
record_sets = dataset.record_sets()
print(f"Available Record Sets (by @id):")
for rs in record_sets:
    print(f"{rs['@id']}: {rs.get('name', '[No name]')}")

# For demonstration, list all fields (by @id) in each record set
all_record_set_fields = {}
for rs in record_sets:
    fields = dataset.fields(record_set=rs["@id"])
    print(f"\nFields for record set '{rs['@id']}' (name: '{rs.get('name','')}'):")
    for field in fields:
        print(f"  Field @id: {field['@id']}  |  name: {field.get('name','')}  |  dataType: {field.get('dataType','')}")
    all_record_set_fields[rs["@id"]] = fields

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# We'll use the first record set for analysis as an example
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Let's select the first populated record set for demonstration
demo_record_set_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        demo_record_set_id = rsid
        break

if demo_record_set_id is None:
    raise Exception("No populated record sets found.")

print(f"Using record set: {demo_record_set_id}")
print("Columns (fields by @id):", dataframes[demo_record_set_id].columns.tolist())
dataframes[demo_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Guess or select a numeric field for demo from fields (e.g., columns containing MW, pI, etc. are common in proteomics)

import numpy as np

# Try to find a numeric field automatically
df = dataframes[demo_record_set_id]
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Try coercing to numeric as fallback
    for col in df.columns:
        temp = pd.to_numeric(df[col], errors='coerce')
        if temp.notna().sum() > 0:
            df[col] = temp
            numeric_field_id = col
            break

if numeric_field_id is None:
    raise Exception("No numeric field found for analysis.")

print(f"Selected numeric field (@id or column): {numeric_field_id}")

# Filter: select rows where value > threshold (using median as threshold)
threshold = df[numeric_field_id].median() if df[numeric_field_id].notna().sum() else 0
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold} (median):")
print(filtered_df.head())

# Normalize the numeric column
norm_col_name = f"{numeric_field_id}_normalized"
filtered_df[norm_col_name] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nFirst 5 normalized values for {numeric_field_id}:")
print(filtered_df[[numeric_field_id, norm_col_name]].head())

# Group by a non-numeric field, if available
group_field = None
for col in df.columns:
    if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]):
        group_field = col
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean {numeric_field_id} by category ({group_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If group_field exists, boxplot
if group_field:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=df[group_field], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xlabel(group_field)
    plt.xticks(rotation=45, ha='right')
    plt.ylabel(numeric_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
We have successfully loaded and explored the dataset using `mlcroissant`, inspected its schema, viewed available fields, extracted and processed numeric data, and generated basic visualizations. For further analysis, review the Croissant schema for detailed record set and field semantics and consult dataset documentation as needed.